In [1]:
import json
import os
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter, defaultdict
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# Configuración
LOG_FILE = "cowrie.json"
SEED = 42
MAX_SEQ_LEN = 15  # Máximo de comandos por sesión
BATCH_SIZE = 64
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"✅ Entorno listo en {DEVICE}")

✅ Entorno listo en cuda


In [2]:
def limpiar_comando(cmd):
    cmd = str(cmd).lower().strip()
    cmd = re.sub(r'https?://\S+', '<URL>', cmd)
    cmd = re.sub(r'\b(?:\d{1,3}\.){3}\d{1,3}\b', '<IP>', cmd)
    cmd = re.sub(r'\b\d{5,}\b', '<NUM>', cmd)
    return cmd[:100]

def etiquetar_ataque(comandos):
    texto = " ".join(comandos).lower()
    if any(x in texto for x in ["btc", "rm -rf", "dd if=", "iptables -f"]): return "G6 - Destrucción/Ransomware"
    if any(x in texto for x in ["tar ", "zip ", "mysqldump", "exfil"]): return "G5 - Robo de Datos"
    if any(x in texto for x in ["useradd", "crontab", "authorized_keys"]): return "G3 - Backdoor/Persistencia"
    if any(x in texto for x in ["wget", "curl", "tftp", "xmrig"]): return "G2 - Malware/Botnet"
    if any(x in texto for x in ["sudo -l", "gcc", "chmod 777"]): return "G4 - Escalada Privilegios"
    return "G1 - Reconocimiento"

# Carga de JSONL
eventos = []
if os.path.exists(LOG_FILE):
    with open(LOG_FILE, "r") as f:
        for linea in f:
            try: eventos.append(json.loads(linea))
            except: pass

# Agrupar por sesión
sesiones_raw = defaultdict(list)
for e in eventos:
    if e.get("eventid") == "cowrie.command.input":
        sesiones_raw[e.get("session")].append(limpiar_comando(e.get("input", "")))

# Crear DataFrame
data = []
for s_id, cmds in sesiones_raw.items():
    if len(cmds) >= 2: # Solo sesiones con comandos reales
        data.append({"session": s_id, "sequence": cmds, "label": etiquetar_ataque(cmds)})

df = pd.DataFrame(data)
print(f"📊 Dataset creado: {len(df)} sesiones de ataque real.")
print(df['label'].value_counts())
df.head()

📊 Dataset creado: 562 sesiones de ataque real.
label
G1 - Reconocimiento            305
G2 - Malware/Botnet             83
G6 - Destrucción/Ransomware     61
G5 - Robo de Datos              54
G4 - Escalada Privilegios       30
G3 - Backdoor/Persistencia      29
Name: count, dtype: int64


,session,sequence,label
0,98bc8025d904,"[ls -la, sudo -l, find / -perm -4000 -type f ...",G1 - Reconocimiento
1,75712e8c6b5f,"[cat /var/www/html/wp-config.php, grep -i 'p...",G1 - Reconocimiento
2,01588324bd47,"[who, enable, system, shell, sh, date, cat /p...",G1 - Reconocimiento
3,83dcc1f75f60,"[env, set, , echo $path, cd /tmp, history, , c...",G1 - Reconocimiento
4,cc1a4494dc86,[mysqldump -u root -p123456 db_name > dump.sql...,G5 - Robo de Datos


In [3]:
# Vocabulario de comandos
all_cmds = [cmd for seq in df['sequence'] for cmd in seq]
vocab = {cmd: i+2 for i, (cmd, _) in enumerate(Counter(all_cmds).most_common(500))}
vocab["<PAD>"], vocab["<UNK>"] = 0, 1

# Mapeo de etiquetas
labels_map = {l: i for i, l in enumerate(df['label'].unique())}
inv_labels_map = {i: l for l, i in labels_map.items()}

def encode_sequence(seq):
    enc = [vocab.get(c, 1) for c in seq][-MAX_SEQ_LEN:]
    return [0] * (MAX_SEQ_LEN - len(enc)) + enc

X = np.array([encode_sequence(s) for s in df['sequence']])
y = np.array([labels_map[l] for l in df['label']])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=SEED)

class HoneypotDataset(Dataset):
    def __init__(self, X, y):
        self.X, self.y = torch.tensor(X, dtype=torch.long), torch.tensor(y, dtype=torch.long)
    def __len__(self): return len(self.X)
    def __getitem__(self, idx): return self.X[idx], self.y[idx]

train_loader = DataLoader(HoneypotDataset(X_train, y_train), batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(HoneypotDataset(X_test, y_test), batch_size=BATCH_SIZE)

In [4]:
class AttackClassifier(nn.Module):
    def __init__(self, vocab_size, num_classes):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, 64, padding_idx=0)
        self.lstm = nn.LSTM(64, 128, num_layers=2, batch_first=True, dropout=0.3)
        self.fc = nn.Linear(128, num_classes)
    
    def forward(self, x):
        _, (h, _) = self.lstm(self.emb(x))
        return self.fc(h[-1])

model = AttackClassifier(len(vocab), len(labels_map)).to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

print("🚀 Entrenando modelo...")
for epoch in range(20):
    model.train()
    for batch_X, batch_y in train_loader:
        batch_X, batch_y = batch_X.to(DEVICE), batch_y.to(DEVICE)
        loss = criterion(model(batch_X), batch_y)
        optimizer.zero_grad(); loss.backward(); optimizer.step()
    if epoch % 5 == 0: print(f"Epoch {epoch} completada")

print("✅ Entrenamiento finalizado.")

🚀 Entrenando modelo...
Epoch 0 completada
Epoch 5 completada
Epoch 10 completada
Epoch 15 completada
✅ Entrenamiento finalizado.


In [5]:
model.eval()
y_pred, y_true = [], []
with torch.no_grad():
    for b_X, b_y in test_loader:
        outputs = model(b_X.to(DEVICE))
        y_pred.extend(outputs.argmax(1).cpu().numpy())
        y_true.extend(b_y.numpy())

print("\n📋 INFORME DE SEGURIDAD (Métricas del Modelo):")
print(classification_report(y_true, y_pred, target_names=list(labels_map.keys())))


📋 INFORME DE SEGURIDAD (Métricas del Modelo):
                             precision    recall  f1-score   support

        G1 - Reconocimiento       1.00      0.98      0.99        61
         G5 - Robo de Datos       1.00      0.91      0.95        11
G6 - Destrucción/Ransomware       0.92      1.00      0.96        12
        G2 - Malware/Botnet       0.94      1.00      0.97        17
 G3 - Backdoor/Persistencia       1.00      1.00      1.00         6
  G4 - Escalada Privilegios       1.00      1.00      1.00         6

                   accuracy                           0.98       113
                  macro avg       0.98      0.98      0.98       113
               weighted avg       0.98      0.98      0.98       113

